In [1]:
import os
import pandas as pd
import glob
from datetime import datetime
import gzip
import tarfile
import shutil

### Definitions

In [2]:
# Function to filter each CSV file
def filter_csv(gz_file, prefixes, region):
	# Read the CSV file into a pandas DataFrame
	data = pd.read_csv(gz_file, dtype=data_types, sep="|")
	
	# Print column names for debugging
	print(f"Column names in {gz_file}: {', '.join(data.columns)}")
	
	# Apply the filter to keep rows where the origin and destination columns start with specified prefixes
	filtered_data = data[
		data["origen"].str.startswith(tuple(prefixes)) &
		data["destino"].str.startswith(tuple(prefixes))
	]
	
	# Define the output file name using the gz_file path
	output_file = os.path.splitext(os.path.basename(gz_file))[0] + "_" + region + ".csv"
	
	# Write the filtered data to a new CSV file
	filtered_data.to_csv(os.path.join(filtered_folder, output_file), index=False)



# Function to aggregate days
def aggregate_days(filtered_folder, criteria, output_file,):
	# Get a list of all CSV files in the folder
	file_list = glob.glob(os.path.join(filtered_folder, "*.csv"))
	# csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
	
	# Initialize an empty list to store data frames
	data_list = []
	
	# Loop through each CSV file
	for file in file_list:
		# Extract the date part from the filename assuming format YYYYMMDD
		file_name = os.path.basename(file)
		date_str = file_name[:8]  # Assuming the first 8 characters are the date
		# year_month = date_str[:6]  # Extract YYYYMM
		
		# Convert thhe date_str to a datetime object
		file_date = datetime.strptime(date_str, "%Y%m%d")

		# Create a weekday flag (Monday=0, Sunday=6)
		weekday_flag = 1 if file_date.weekday() < 5 else 0

		# Read the data
		data = pd.read_csv(file, dtype=data_types, sep=",")
		data['date_str'] = date_str
		data['weekday_flag'] = weekday_flag
		data_list.append(data)
		
		# Print the total number of trips each day
		print(f"{file}: {data['viajes'].sum()}")
	
	# Combine all data frames into one
	combined_data = pd.concat(data_list, ignore_index=True)
	print(f"Total trips: {combined_data['viajes'].sum()}")

	grouping_criteria = criteria + ['date_str', 'weekday_flag']
	
	# The following line fills any NaN before grouping to prevent rows from being lost
	combined_data[grouping_criteria] = combined_data[grouping_criteria].fillna("Unknown")

	# Group by criteria and calculate the sums
	grouped_data = combined_data.groupby(grouping_criteria).agg(
		viajes=('viajes', 'sum'),
		viajes_km=('viajes_km', 'sum')
	).reset_index()

	print(f"Output trips({output_file}): {grouped_data['viajes'].sum()}")
	print(f"Output trip-kilometers({output_file}): {grouped_data['viajes_km'].sum()}")

	# Write the filtered data to a new CSV file
	grouped_data.drop_duplicates(inplace = True)
	grouped_data.to_csv(os.path.join(output_folder, output_file), index=False)


### Pseudo main

In [ ]:
# Define working folder. We'll work in parallel with the 3 possible subdivisions (GAU, municipality, census district)
working_folder = "D:/Documents/Proyectos/MITMA/Data/"

# Define sub-folders
tar_file_path = os.path.join(working_folder, "0-meses_completos/")
gz_file_path = os.path.join(working_folder, "1-Unzipped_meses_completos/")
raw_files = os.path.join(working_folder, "1-Unzipped/")
filtered_folder = os.path.join(working_folder,"2-Ourense/")
if not os.path.exists(filtered_folder):
	os.makedirs(filtered_folder)
output_folder = os.path.join(working_folder,"3-Results/")
zones_shp = os.path.join(working_folder, "Zonificación_GAUs/")
# output_folder = os.path.join(working_folder, "3-Averages/")

# Create a temporary folder to store the .gz files inside each .tar file
tmp_folder = os.path.join(working_folder, "1-tmp")
if os.path.exists(tmp_folder):
	shutil.rmtree(tmp_folder)
	os.makedirs(tmp_folder)
else:
	os.makedirs(tmp_folder)

# Define datatypes
data_types = {"origen": 'string',
			  "destino": 'string', 
			  "distancia": 'string',
			  "actividad_origen": 'string',
			  "actividad_destino": 'string',
			  "residencia": 'string',
			  "renta": 'string',
			  "edad": 'string',
			  "sexo": 'string'
}

criteria = list(data_types.keys())

# Define columns with quantities to group
prefixes = ["32"]
region = "Ourense"

tar_pattern = os.path.join(tar_file_path, "*.tar")
tar_file_list = glob.glob(tar_pattern)
pattern = os.path.join(raw_files, "*.csv")
file_list = glob.glob(pattern)
output_file = region + "_MITMA.csv"

list_dates = ['20230128','20230129','20230225','20230226','20230311','20230312', '20230422','20230423',
			'20230513','20230514', '20230610','20230611', '20230722','20230723', '20230812','20230813',
			'20230909','20230910', '20231007', '20231008', '20231111','20231112', '20231216','20231217'
			]

# list_dates = ['20230124','20230125','20230126','20230221','20230222','20230223','20230307','20230308','20230309',
# 			'20230418','20230419','20230420', '20230509','20230510','20230511', '20230606','20230607','20230608',
# 			'20230718','20230719','20230720', '20230808','20230809','20230810', '20230905','20230906','20230907',
# 			'20231003', '20231004','20231005','20231107','20231108', '20231109','20231212','20231213','20231214',
# 			]

# Apply the filter function to each .tar file
for tar_file in tar_file_list:
	with tarfile.open(tar_file, 'r') as tar:
		tar.extractall(path=tmp_folder)
		for gz_file in tar.getmembers():
			if gz_file.name.endswith('.gz'):
				if any(date in gz_file.name for date in list_dates):
					gz_file_path = os.path.join(tmp_folder, gz_file.name)
					filter_csv(gz_file_path, prefixes, region)

		# Clear the tmp_folder after processing each tar file
		shutil.rmtree(tmp_folder)
		os.makedirs(tmp_folder)  # Recreate the tmp folder


Column names in D:/Documents/Proyectos/MITMA/Data/1-tmp\20230128_Viajes_distritos.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in D:/Documents/Proyectos/MITMA/Data/1-tmp\20230129_Viajes_distritos.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in D:/Documents/Proyectos/MITMA/Data/1-tmp\20230225_Viajes_distritos.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in D:/Documents/Proyectos/MITMA/Data/1-tmp\20230226_Viajes_distritos.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible

In [4]:
aggregate_days(filtered_folder, criteria, output_file)


D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230124_Viajes_distritos.csv_Ourense.csv: 856644.0519999998
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230125_Viajes_distritos.csv_Ourense.csv: 861042.142
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230126_Viajes_distritos.csv_Ourense.csv: 848605.9139999999
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230128_Viajes_distritos.csv_Ourense.csv: 732993.476
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230129_Viajes_distritos.csv_Ourense.csv: 662981.74
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230221_Viajes_distritos.csv_Ourense.csv: 705172.3647167153
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230222_Viajes_distritos.csv_Ourense.csv: 810683.0650000002
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230223_Viajes_distritos.csv_Ourense.csv: 858252.771
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230225_Viajes_distritos.csv_Ourense.csv: 713468.3820000001
D:/Documents/Proyectos/MITMA/Data/2-Ourense\20230226_Viajes_distritos.csv_Ourense.cs

### Date substitution routine

file1 = "D:/Documents/Proyectos/MITMA/Data/3-Results/Leon_Mitma.csv"
file2 = "D:/Documents/Proyectos/MITMA/Data/3-Results/Leon_Mitma_week_Oct.csv"
output = "D:/Documents/Proyectos/MITMA/Data/3-Results/Leon_Mitma_new.csv"

dates = ['20231002',
         '20231003',
         '20231004',
         '20231005',
         '20231006',
         '20231007',
         '20231008']


df1 = pd.read_csv(file1)
df1['date_str'] = df1['date_str'].astype(str)
df1.drop(df1[df1['date_str'].isin(dates)].index, inplace=True)
df2 = pd.read_csv(file2)
df1 = pd.concat([df1, df2])
df1.to_csv(output, index=False)